# Przegląd wyjaśnień modelu z zautomatyzowanego uczenia

Modele uczenia maszynowego decydują dziś o coraz poważniejszych sprawach - banki opierają na nich decyzje kredytowe, a w medycynie wspierają ustalanie kolejności leczenia. Dlatego rośnie znaczenie **interpretowalności** (ang. *interpretability*): trzeba umieć wytłumaczyć i uzasadnić, dlaczego model przewidział to, co przewidział, a także wychwycić niezamierzone uprzedzenia (ang. *bias*) ukryte w danych.

Przy zautomatyzowanym uczeniu maszynowym wystarczy ustawić `enable_model_explainability=True`, a Azure Machine Learning wyliczy dla najlepszego modelu **istotność cech** (ang. *feature importance*) - miarę tego, jak mocno każda kolumna wpływa na przewidywania. Wyniki obejrzysz w Studio na karcie **Explanations (preview)**.

> **A co z pulpitem odpowiedzialnej sztucznej inteligencji**: pełny **pulpit RAI** (ang. *Responsible AI dashboard*) - z analizą błędów, sprawiedliwością i analizą przyczynową - **nie jest dostępny dla modeli AutoML**. Studio pokazuje w tym miejscu komunikat „Responsible AI dashboard is currently not supported for AutoML models", a przycisk tworzenia pulpitu jest nieaktywny. Dla modeli AutoML zostają wyjaśnienia opisane w tym ćwiczeniu. Pełny pulpit zbudujesz w [ćwiczeniu 9B](labdocs/Lab09B.md), dla modelu wytrenowanego zwykłym skryptem.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

ml_client = MLClient.from_config(
    credential=credential
)

print(f"Polaczono z obszarem roboczym: {ml_client.workspace_name}")

## Uruchomienie zadania zautomatyzowanego uczenia

Żeby skrócić czas ćwiczenia, uruchomisz zadanie na klastrze **aml-cluster** z niewielkim budżetem prób.

Zwróć uwagę na `enable_model_explainability` ustawione na `True` - dzięki temu Azure Machine Learning wyliczy dla najlepszego modelu istotność cech.

In [ ]:
from azure.ai.ml import automl, Input
from azure.ai.ml.constants import AssetTypes

# Pobieramy zarejestrowany zasob danych treningowych - wersja "latest" sprawia,
# ze kod zadziala niezaleznie od tego, ktora wersje zasobu diabetes_mltable
# (typu mltable) masz aktualnie zarejestrowana.
diabetes_data_asset = ml_client.data.get(name="diabetes_mltable", label="latest")

# Konfigurujemy zadanie klasyfikacji AutoML
classification_job = automl.classification(
    compute="aml-cluster",
    experiment_name="diabetes-automl",
    training_data=Input(type=AssetTypes.MLTABLE, path=diabetes_data_asset.id),
    target_column_name="Diabetic",
    primary_metric="AUC_weighted",
    n_cross_validations=2,
)

# Ustawiamy limity zadania - ile prob i jak dlugo moga trwac
classification_job.set_limits(
    max_trials=3,
    max_concurrent_trials=3,
    timeout_minutes=30,
)

# Bez przetwarzania cech - model dostaje kolumny takie, jakie są w danych.
# Za chwilę uruchomisz to samo zadanie z przetwarzaniem i porównasz wyniki.
classification_job.set_featurization(mode="off")

# Wlaczamy wyjasnianie modelu - dla najlepszego modelu powstanie
# istotnosc cech. Zespoly modeli wylaczamy: wymagaja
# co najmniej czterech prob, a tutaj budzet jest celowo maly.
classification_job.set_training(
    enable_model_explainability=True,
    enable_stack_ensemble=False,
    enable_vote_ensemble=False,
)

# Wysylamy zadanie AutoML
returned_job = ml_client.jobs.create_or_update(classification_job)
print(f"Wyslano zadanie: {returned_job.name}")

# Strumieniujemy dziennik zadania az do jego zakonczenia
ml_client.jobs.stream(returned_job.name)

## Przegląd istotności cech

Po zakończeniu zadania otwórz je w [Azure Machine Learning studio](https://ml.azure.com) - poniższa komórka wypisze bezpośredni odnośnik. Przejdź na kartę **Models + child jobs**, wybierz model o najwyższym wyniku i otwórz kartę **Explanations (preview)**. Zobaczysz tam względną istotność poszczególnych cech.

> **Jeśli karta jest pusta**: zaznacz model i kliknij **Explain model**, wskazując klaster obliczeniowy. Uruchomi to wyliczanie wyjaśnień jako osobne zadanie podrzędne - trwa kilka minut, po czym karta się wypełni.

In [ ]:
# Pobieramy zakonczone zadanie i wypisujemy odnosnik do jego podgladu w Studio
completed_job = ml_client.jobs.get(returned_job.name)
print(f"Stan: {completed_job.status}")
print(f"Adres w Studio: {completed_job.services['Studio'].endpoint}")

## Istotność cech wytworzonych automatycznie

Poprzedni przebieg trenował na surowych kolumnach. Zautomatyzowane uczenie maszynowe potrafi jednak samo przetwarzać dane przed trenowaniem - wykonuje **inżynierię cech** (ang. *feature engineering*), czyli tworzy nowe kolumny wyliczone z tych, które już są. Włączysz teraz tę opcję metodą `set_featurization` i uruchomisz zadanie ponownie.

> **Po co drugi przebieg**: chodzi o porównanie. Za chwilę zobaczysz w Studio przełącznik między cechami surowymi a wytworzonymi - i przekonasz się, czy automat wymyślił coś, co realnie pomaga, czy tylko rozmnożył kolumny.

> **Jedna kolumna wymaga poprawki**: `Pregnancies` zawiera liczby, ale automat uznaje ją za kategorię - ma niewiele różnych wartości. Rozwija ją wtedy w kilkanaście rzadkich cech zakodowanych tekstowo, co zniekształca istotność cech - kilkanaście sztucznych kolumn zamiast jednej prawdziwej. Dlatego typ tej kolumny wskazujemy wprost. To dobry przykład na to, że automatyzacja potrzebuje czasem korekty od kogoś, kto wie, co te dane znaczą.

In [ ]:
from azure.ai.ml import automl, Input
from azure.ai.ml.constants import AssetTypes

# Konfigurujemy zadanie klasyfikacji AutoML - tym razem z wlaczonym
# przetwarzaniem cech
classification_job = automl.classification(
    compute="aml-cluster",
    experiment_name="diabetes-automl",
    training_data=Input(type=AssetTypes.MLTABLE, path=diabetes_data_asset.id),
    target_column_name="Diabetic",
    primary_metric="AUC_weighted",
    n_cross_validations=2,
)

classification_job.set_limits(
    max_trials=3,
    max_concurrent_trials=3,
    timeout_minutes=30,
)

# Tym razem z przetwarzaniem cech. Pregnancies to liczba, ale AutoML wykrywa ją
# jako kategorię i rozwija w kilkanaście rzadkich cech tekstowych - co zniekształca
# istotność cech.
classification_job.set_featurization(
    mode="auto",
    column_name_and_types={"Pregnancies": "Numeric"},
)

# Wlaczamy wyjasnianie modelu - dla najlepszego modelu powstanie
# istotnosc cech. Zespoly modeli wylaczamy: wymagaja
# co najmniej czterech prob, a tutaj budzet jest celowo maly.
classification_job.set_training(
    enable_model_explainability=True,
    enable_stack_ensemble=False,
    enable_vote_ensemble=False,
)

# Wysylamy zadanie AutoML
returned_job = ml_client.jobs.create_or_update(classification_job)
print(f"Wyslano zadanie: {returned_job.name}")

ml_client.jobs.stream(returned_job.name)

Przetwarzanie cech realizują [potoki przekształceń scikit-learn](https://scikit-learn.org/stable/modules/compose.html#combining-estimators) - nie mylić z potokami Azure Machine Learning. Powstały model zawiera więc także kroki przygotowania danych, wykonywane przed każdą predykcją.

Uruchom poniższy kod, żeby uzyskać odnośnik do tego zadania w Studio. Na karcie **Explanations (preview)** znajdź przełącznik **Raw features** / **Engineered features** i porównaj istotność oryginalnych kolumn z istotnością cech wytworzonych automatycznie.

In [ ]:
# Pobieramy zakonczone zadanie i wypisujemy odnosnik do jego podgladu w Studio
completed_job = ml_client.jobs.get(returned_job.name)
print(f"Stan: {completed_job.status}")
print(f"Adres w Studio: {completed_job.services['Studio'].endpoint}")

> **Więcej informacji**: o zautomatyzowanym uczeniu maszynowym przeczytasz w [dokumentacji Azure ML](https://learn.microsoft.com/azure/machine-learning/how-to-configure-auto-train), a o interpretowalności modeli - w artykule [Model interpretability](https://learn.microsoft.com/azure/machine-learning/how-to-machine-learning-interpretability).